In [ ]:
# Import all necessary dependencies
import os
import pandas as pd
from azure.storage.blob import BlobServiceClient, BlobClient, ContainerClient
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Data Extraction
try:
    data = pd.read_csv(r"dataset/raw_data/zipco_transaction.csv")
    print("Data loaded successfully!")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# Data cleaning and transformation
# Remove duplicates
data.drop_duplicates(inplace=True)

# Handling missing values (Example: fill missing numeric values with the mean or median)
numeric_columns = data.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_columns:
    #data[col].fillna(data[col].mean(), inplace=True)
    data.fillna({col: data[col].mean()}, inplace=True)

In [ ]:
# Handling missing values (Example: fill missing string values with 'Unknown')
string_columns = data.select_dtypes(include=['object']).columns
for col in string_columns:
    #data[col].fillna('Unknown', inplace=True)
    data.fillna({col: 'Unknown'}, inplace=True)

In [ ]:
data.info()

In [ ]:
# Create Products Table
products = data[['ProductName', 'UnitPrice']].drop_duplicates().reset_index(drop=True)
products.index.name = 'ProductID'
products = products.reset_index()

In [ ]:
# Create Customers Table
customers = data[['CustomerName', 'CustomerAddress', 'Customer_PhoneNumber', 'CustomerEmail']].drop_duplicates().reset_index(drop=True)
customers.index.name = 'CustomerID'
customers = customers.reset_index()

In [ ]:
# Create Staff Table
staff = data[['Staff_Name', 'Staff_Email']].drop_duplicates().reset_index(drop=True)
staff.index.name = 'StaffID'
staff = staff.reset_index()

In [ ]:
# Create Transaction Table
transactions = data.merge(products, on = ['ProductName', 'UnitPrice'], how='left') \
                   .merge(customers, on = ['CustomerName', 'CustomerAddress', 'Customer_PhoneNumber', 'CustomerEmail'], how='left') \
                   .merge(staff, on= ['Staff_Name', 'Staff_Email'], how='left')
transactions.index.name = 'TransactionID'
transactions = transactions.reset_index() \
                           [['TransactionID', 'Date', 'ProductID', 'CustomerID', 'StaffID', 'Quantity', 'StoreLocation', 'PaymentType', \
                                'PromotionApplied', 'Weather', 'Temperature', 'StaffPerformanceRating', 'CustomerFeedback', \
                                'DeliveryTime_min', 'OrderType', 'DayOfWeek', 'TotalSales']]

In [ ]:
transactions.head()

In [ ]:
# Save normalized tables to new CSV files
data.to_csv('dataset/cleaned_data/clean_data.csv', index=False)
products.to_csv('dataset/cleaned_data/products.csv', index=False)
customers.to_csv('dataset/cleaned_data/customers.csv', index=False)
staff.to_csv('dataset/cleaned_data/staff.csv', index=False)
transactions.to_csv('dataset/cleaned_data/transactions.csv', index=False)

In [ ]:
# Azure connection string and container name
# Load environment variables from .env file
connect_str = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
container_name = os.getenv('CONTAINER_NAME')

# Create a BlobServiceClient object
blob_service_client = BlobServiceClient.from_connection_string(connect_str)
container_client = blob_service_client.get_container_client(container_name)

# Load data to Azure Blob Storage
# List of tuples (DataFrame, Blob Name)
files = [
    (data, "raw/cleaned_zipco_transaction_data.csv"),
    (products, "processed/products.csv"),
    (customers, "processed/customers.csv"),
    (staff, "processed/staff.csv"),
    (transactions, "processed/transactions.csv")
]

# Load data to Azure Blob Storage
for file, blob_name in files:
    blob_client = container_client.get_blob_client(blob_name)
    output = file.to_csv(index=False)
    blob_client.upload_blob(output, overwrite=True)
    print(f"{blob_name} loaded into Azure Blob Storage.")